<a href="https://colab.research.google.com/github/furkanaras0/Tez-recipe-recommender-system/blob/main/FoodcomTfidf3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. KURULUM VE KÜTÜPHANELER
# ==============================================================================
!pip install git+https://github.com/lyst/lightfm.git

import numpy as np
import pandas as pd
import ast
import os
import pickle
import scipy.sparse as sp
from google.colab import drive
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightfm import LightFM
from lightfm.data import Dataset

DATA_PATH = "/content"
NUM_THREADS = os.cpu_count()

  Cloning https://github.com/lyst/lightfm.git to /tmp/pip-req-build-uj_40nmz
  Running command git clone --filter=blob:none --quiet https://github.com/lyst/lightfm.git /tmp/pip-req-build-uj_40nmz
  Resolved https://github.com/lyst/lightfm.git to commit 0c9c31e027b976beab2385e268b58010fff46096
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=833397 sha256=aadade8b9071116a19cf02536c4584c7a42bc464c3cde74b5827d449de36c794
  Stored in directory: /tmp/pip-ephem-wheel-cache-86rjtdm9/wheels/a1/19/87/4ce72d7fa398c33a08d5c94fdb76c4e24f4c95f9a3c230b4b6
Successfully built lightfm


In [ ]:
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/Tez/food-com-recipes-and-user-interactions.zip -d /content/

Mounted at /content/drive


In [ ]:
# ==============================================================================
# 2. VERİ YÜKLEME VE TEMİZLİK (10-CORE FİLTRELEME)
# ==============================================================================
print("🔄 Veriler yükleniyor...")

interactions = pd.read_csv(
    DATA_PATH + "/RAW_interactions.csv",
    usecols=["user_id", "recipe_id", "rating"],
    dtype={"user_id": "int32", "recipe_id": "int32", "rating": "float32"}
)

recipes = pd.read_csv(
    DATA_PATH + "/RAW_recipes.csv",
    dtype={"id": "int32"},
    engine='python',
    on_bad_lines='skip'
)

# Sadece pozitif rating (Implicit sinyal) ve Null/Duplicate temizliği
interactions = interactions[interactions["rating"] > 0].drop_duplicates(subset=["user_id", "recipe_id"], keep="last").dropna()
recipes = recipes.dropna(subset=['id', 'name'])

# Karşılıklı Tutarlılık Kontrolü
valid_recipe_ids = set(recipes["id"].unique())
interactions = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].reset_index(drop=True)

print("🔄 10-Core Filtreleme uygulanıyor (Kullanıcı ve Tarif için döngüsel temizlik)...")

# 10-Core İteratif Filtreleme Döngüsü
min_interactions = 10
while True:
    start_len = len(interactions)

    # 1. Kullanıcıları filtrele
    user_counts = interactions['user_id'].value_counts()
    valid_users = user_counts[user_counts >= min_interactions].index
    interactions = interactions[interactions['user_id'].isin(valid_users)]

    # 2. Tarifleri filtrele
    recipe_counts = interactions['recipe_id'].value_counts()
    valid_recipes = recipe_counts[recipe_counts >= min_interactions].index
    interactions = interactions[interactions['recipe_id'].isin(valid_recipes)]

    # Eğer bu turda hiçbir şey elenmediyse veri dengelenmiş demektir, döngüden çık.
    if len(interactions) == start_len:
        break

interactions_filtered = interactions.copy().reset_index(drop=True)

print(f"✅ 10-Core Temizlik Tamamlandı.")
print(f"Kalan Etkileşim Sayısı: {len(interactions_filtered)}")
print(f"Kalan Eşsiz Kullanıcı: {interactions_filtered['user_id'].nunique()}")
print(f"Kalan Eşsiz Tarif: {interactions_filtered['recipe_id'].nunique()}")

🔄 Veriler yükleniyor...
🔄 10-Core Filtreleme uygulanıyor (Kullanıcı ve Tarif için döngüsel temizlik)...
✅ 10-Core Temizlik Tamamlandı.
Kalan Etkileşim Sayısı: 298969
Kalan Eşsiz Kullanıcı: 7196
Kalan Eşsiz Tarif: 12474


In [ ]:
# ==============================================================================
# 3. TRAIN / TEST SPLIT (KULLANICI BAZLI / PER-USER SPLIT)
# ==============================================================================
print("🔄 Veri hazırlanıyor (Kullanıcı Bazlı - Per User Split)...")

# 1. Kullanıcı bazlı stratifikasyon (Her kullanıcının kendi verisinin %80'i Train, %20'si Test)
train_df = interactions_filtered.groupby('user_id', group_keys=False).apply(
    lambda x: x.sample(frac=0.80, random_state=42)
)

# 2. Train setine girmeyen geri kalan tüm satırları Test seti yap
test_df = interactions_filtered.drop(train_df.index)

# 3. Güvenlik (Cold-Start İtem Filtresi):
# Kullanıcıların hepsini eğitime aldık ama şans eseri nadir bir "Tarif" tamamen test setine düşmüş olabilir.
# Sadece eğitimde gördüğümüz tarifleri test setinde tutuyoruz.
train_items = set(train_df["recipe_id"])

test_df = test_df[test_df["recipe_id"].isin(train_items)].reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

# Sadece Train'de olan tariflerin metadatasını al
recipes = recipes[recipes["id"].isin(train_items)].reset_index(drop=True)

print(f"✅ Train Boyutu: {len(train_df)} | Test Boyutu: {len(test_df)}")

🔄 Veri hazırlanıyor (Kullanıcı Bazlı - Per User Split)...
✅ Train Boyutu: 239265 | Test Boyutu: 59704


/tmp/ipython-input-4-308415253.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = interactions_filtered.groupby('user_id', group_keys=False).apply(


In [ ]:
# ==============================================================================
# 4. ÖZELLİK (FEATURE) MÜHENDİSLİĞİ VE TF-IDF
# ==============================================================================
print("🔄 TF-IDF ve Kategorik Özellikler hazırlanıyor...")

def safe_literal_eval(x):
    if pd.isna(x) or x == "": return []
    try: return x if isinstance(x, list) else ast.literal_eval(x)
    except: return []

recipes["tags"] = recipes["tags"].apply(safe_literal_eval)
recipes["ingredients"] = recipes["ingredients"].apply(safe_literal_eval)
recipes["nutrition"] = recipes["nutrition"].apply(safe_literal_eval)

recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce").fillna(0).clip(lower=0)
recipes["time_bucket"] = pd.cut(recipes["minutes"], bins=[-1, 15, 30, 60, float("inf")], labels=["very_fast", "fast", "medium", "long"]).astype(str)

def extract_calories(nut_list):
    try: return float(nut_list[0]) if len(nut_list) > 0 else 0.0
    except: return 0.0
recipes["calories"] = recipes["nutrition"].apply(extract_calories).clip(lower=0)
recipes["calorie_bucket"] = pd.cut(recipes["calories"], bins=[-1, 200, 500, float("inf")], labels=["low_cal", "medium_cal", "high_cal"]).astype(str)

# TF-IDF İçin Metin Birleştirme
recipes["content_text"] = recipes["name"].fillna("") + " " + recipes["tags"].apply(lambda x: " ".join(x)) + " " + recipes["ingredients"].apply(lambda x: " ".join(x))

tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(recipes['content_text'])
tfidf_feature_names = [f"tfidf_{name}" for name in tfidf.get_feature_names_out()]
recipe_id_to_idx = {rid: i for i, rid in enumerate(recipes['id'])}

print(f"✅ TF-IDF Vektörleştirme Tamamlandı. (Boyut: {tfidf_matrix.shape})")

🔄 TF-IDF ve Kategorik Özellikler hazırlanıyor...
✅ TF-IDF Vektörleştirme Tamamlandı. (Boyut: (12474, 5486))


In [ ]:
# ==============================================================================
# 5. LIGHTFM DATASET İNŞASI
# ==============================================================================
print("🔄 LightFM Dataset ve Sparse Matrisler oluşturuluyor...")

dataset = Dataset()
user_ids = sorted(train_df["user_id"].unique())
item_ids = sorted(train_df["recipe_id"].unique())

all_tags = set(f"tag_{x}" for t in recipes["tags"] for x in t)
all_ingredients = set(f"ing_{x}" for ing_list in recipes["ingredients"] for x in ing_list)
all_time = set(f"time_{x}" for x in recipes["time_bucket"].unique())
all_cal  = set(f"cal_{x}" for x in recipes["calorie_bucket"].unique())

item_features_list = sorted(list(all_tags) + list(all_ingredients) + list(all_time) + list(all_cal) + tfidf_feature_names)

dataset.fit(users=user_ids, items=item_ids, item_features=item_features_list)

# Etkileşim Matrisleri
train_df["interaction"] = 1.0
interactions_matrix, _ = dataset.build_interactions(zip(train_df["user_id"], train_df["recipe_id"], train_df["interaction"]))
test_interactions, _ = dataset.build_interactions(zip(test_df["user_id"], test_df["recipe_id"], np.ones(len(test_df))))
test_interactions_csr = test_interactions.tocsr() # Metrikler için CSR formatı

# TF-IDF Ağırlıklı Özellik Matrisi Oluşturma
def build_weighted_item_features(row, tfidf_mat, tfidf_names, rec_to_idx):
    item_id = row["id"]
    feat_dict = {f"tag_{tag}": 1.0 for tag in row["tags"]}
    feat_dict.update({f"ing_{ing}": 1.0 for ing in row["ingredients"]})
    feat_dict[f"time_{row['time_bucket']}"] = 1.0
    feat_dict[f"cal_{row['calorie_bucket']}"] = 1.0

    if item_id in rec_to_idx:
        row_sparse = tfidf_mat[rec_to_idx[item_id]]
        for col_idx, weight in zip(row_sparse.indices, row_sparse.data):
            feat_dict[tfidf_names[col_idx]] = weight
    return (item_id, feat_dict)

item_features_data = [build_weighted_item_features(row._asdict(), tfidf_matrix, tfidf_feature_names, recipe_id_to_idx) for row in recipes.itertuples(index=False)]
item_features_matrix = dataset.build_item_features(item_features_data)

print(f"✅ Matrisler Hazır. Özellik Boyutu: {item_features_matrix.shape}")

# Hızlı erişim (Mapping) sözlükleri
item_id_map = dataset.mapping()[2]
idx_to_recipe_id = {v: k for k, v in item_id_map.items()}

🔄 LightFM Dataset ve Sparse Matrisler oluşturuluyor...
✅ Matrisler Hazır. Özellik Boyutu: (12474, 23395)


In [ ]:
# ==============================================================================
# 6. MODEL EĞİTİMİ
# ==============================================================================
EPOCH_SAYISI = 50
print(f"\n🚀 {EPOCH_SAYISI} Epoch'luk Gelişmiş Model Eğitimi Başlıyor...")

model = LightFM(loss='warp',
                no_components=250,
                learning_rate=0.05,
                random_state=42)

for epoch in tqdm(range(EPOCH_SAYISI), desc="Eğitim İlerlemesi"):
    model.fit_partial(interactions_matrix, item_features=item_features_matrix, epochs=1, num_threads=NUM_THREADS)


🚀 50 Epoch'luk Gelişmiş Model Eğitimi Başlıyor...


Eğitim İlerlemesi: 100%|██████████| 50/50 [12:48<00:00, 15.37s/it]


In [ ]:
# ==============================================================================
# MODEL VE YARDIMCI BİLEŞENLERİN KAYDEDİLMESİ
# ==============================================================================
import pickle
import scipy.sparse as sp
import os

# Kayıt klasörünü belirle
SAVE_DIR = '/content/drive/MyDrive/Tez3/Saved_Models_TFİDF3/'
os.makedirs(SAVE_DIR, exist_ok=True)

print("💾 Kayıt işlemi başlatılıyor, lütfen bekleyin...")

try:
    # Ufak isim uyuşmazlığını çözen o sihirli satır:
    recipe_id_to_tfidf_idx = recipe_id_to_idx

    # 1. LightFM Modelini ve Dataset Yapısını Kaydet (Pickle)
    with open(SAVE_DIR + 'lightfm_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open(SAVE_DIR + 'lightfm_dataset.pkl', 'wb') as f:
        pickle.dump(dataset, f)

    # 2. Sparse (Seyrek) Matrisleri Kaydet (.npz)
    sp.save_npz(SAVE_DIR + 'interactions_matrix.npz', interactions_matrix)
    sp.save_npz(SAVE_DIR + 'item_features_matrix.npz', item_features_matrix)
    sp.save_npz(SAVE_DIR + 'tfidf_matrix.npz', tfidf_matrix)

    # 3. test_interactions matrisini kaydet (Eğer csr formatında değilse çevirip kaydet)
    if not isinstance(test_interactions, sp.csr_matrix):
        test_interactions_csr = test_interactions.tocsr()
    else:
        test_interactions_csr = test_interactions
    sp.save_npz(SAVE_DIR + 'test_interactions_csr.npz', test_interactions_csr)

    # 4. TF-IDF Nesnesini Kaydet
    with open(SAVE_DIR + 'tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(tfidf, f)

    # 5. İşlenmiş Veri Çerçevelerini Kaydet
    recipes.to_pickle(SAVE_DIR + 'recipes_processed.pkl')
    test_df.to_pickle(SAVE_DIR + 'test_df.pkl')

    # 6. Mapping Sözlüklerini Kaydet
    mappings = {
        'idx_to_recipe_id': idx_to_recipe_id,
        'recipe_id_to_tfidf_idx': recipe_id_to_tfidf_idx
    }
    with open(SAVE_DIR + 'mappings.pkl', 'wb') as f:
        pickle.dump(mappings, f)

    print(f"\n✅ BAŞARILI: Tüm bileşenler şu konuma kaydedildi:\n📍 {SAVE_DIR}")

except Exception as e:
    print(f"❌ Kayıt sırasında bir hata oluştu: {e}")

💾 Kayıt işlemi başlatılıyor, lütfen bekleyin...

✅ BAŞARILI: Tüm bileşenler şu konuma kaydedildi:
📍 /content/drive/MyDrive/Tez3/Saved_Models_TFİDF3/


In [ ]:
# ==============================================================================
# 7. METRİK DEĞERLENDİRME
# ==============================================================================
print("\n🧪 METRİKLER HESAPLANIYOR (Bu işlem biraz zaman alabilir)...\n")

# A. Temel Metrikler (Precision, Recall, F1, AUC ve NDCG)
def evaluate_core_metrics(model, train, test, features, k=10, batch_size=200, sample_size=10000):
    train, test = train.tocsr(), test.tocsr()
    n_users, n_items = train.shape
    precisions, recalls, aucs, ndcgs = [], [], [], []
    valid_users = np.where(test.getnnz(axis=1) > 0)[0]

    if len(valid_users) > sample_size:
        np.random.seed(42)
        valid_users = np.random.choice(valid_users, sample_size, replace=False)

    for start in tqdm(range(0, len(valid_users), batch_size), desc="Temel Metrikler (NDCG Dahil)"):
        batch = valid_users[start:start+batch_size]
        scores_batch = model.predict(np.repeat(batch, n_items), np.tile(np.arange(n_items), len(batch)), item_features=features, num_threads=NUM_THREADS).reshape(len(batch), n_items)

        for i, u in enumerate(batch):
            scores = scores_batch[i].copy()
            tr_idx, te_idx = train[u].indices, test[u].indices
            if len(te_idx) == 0: continue

            scores[tr_idx] = -np.inf # Masking
            top_k = np.argpartition(-scores, k-1)[:k]

            # Kendi içinde sıralama (NDCG için zorunlu adım)
            top_k_sorted = top_k[np.argsort(-scores[top_k])]

            # Precision ve Recall
            hits = len(set(top_k_sorted).intersection(set(te_idx)))
            precisions.append(hits / k)
            recalls.append(hits / len(te_idx))

            # --- NDCG HESAPLAMASI ---
            hits_mask = np.isin(top_k_sorted, te_idx) # Önerilenler arasında hangileri doğru?
            dcg = np.sum(hits_mask / np.log2(np.arange(2, k + 2)))
            idcg = np.sum(1.0 / np.log2(np.arange(2, min(len(te_idx), k) + 2)))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            ndcgs.append(ndcg)

            # AUC
            valid_mask = np.ones(n_items, dtype=bool)
            valid_mask[tr_idx] = False
            y_true = np.zeros(n_items)
            y_true[te_idx] = 1
            try: aucs.append(roc_auc_score(y_true[valid_mask], scores[valid_mask]))
            except: continue

    p, r = np.mean(precisions), np.mean(recalls)
    return p, r, (2*p*r)/(p+r) if (p+r)>0 else 0, np.mean(aucs), np.mean(ndcgs)

precision, recall, f1, auc, ndcg = evaluate_core_metrics(model, interactions_matrix, test_interactions_csr, item_features_matrix)

# B. Diversity ve Coverage
def evaluate_diversity_coverage(model, train, test, tfidf_mat, features, rec_to_tfidf, idx_to_rec, k=10, batch_size=200, sample_size=10000):
    train, test = train.tocsr(), test.tocsr()
    n_users, n_items = train.shape
    diversities, recommended_items = [], set()
    valid_users = np.where(test.getnnz(axis=1) > 0)[0]
    if len(valid_users) > sample_size: np.random.seed(42); valid_users = np.random.choice(valid_users, sample_size, replace=False)

    for start in tqdm(range(0, len(valid_users), batch_size), desc="Div & Cov"):
        batch = valid_users[start:start+batch_size]
        scores_batch = model.predict(np.repeat(batch, n_items), np.tile(np.arange(n_items), len(batch)), item_features=features, num_threads=NUM_THREADS).reshape(len(batch), n_items)

        for i, u in enumerate(batch):
            scores = scores_batch[i].copy()
            scores[train[u].indices] = -np.inf
            top_k = np.argpartition(-scores, k-1)[:k]
            recommended_items.update(top_k)

            p_tfidf_idx = [rec_to_tfidf[idx_to_rec[idx]] for idx in top_k if idx_to_rec[idx] in rec_to_tfidf]
            if len(p_tfidf_idx) > 1:
                sim_matrix = cosine_similarity(tfidf_mat[p_tfidf_idx])
                diversities.append(1 - np.mean(sim_matrix[np.triu_indices(len(p_tfidf_idx), k=1)]))

    return np.mean(diversities), len(recommended_items) / n_items

diversity, coverage = evaluate_diversity_coverage(model, interactions_matrix, test_interactions_csr, tfidf_matrix, item_features_matrix, recipe_id_to_tfidf_idx, idx_to_recipe_id)

# C. HD95
def evaluate_hd95(model, train, test, tfidf_mat, features, rec_to_tfidf, idx_to_rec, k=10, batch_size=200, sample_size=10000):
    train, test = train.tocsr(), test.tocsr()
    n_items = train.shape[1]
    hd_scores = []
    valid_users = np.where(test.getnnz(axis=1) > 0)[0]
    if len(valid_users) > sample_size: np.random.seed(42); valid_users = np.random.choice(valid_users, sample_size, replace=False)

    for start in tqdm(range(0, len(valid_users), batch_size), desc="HD95"):
        batch = valid_users[start:start+batch_size]
        scores_batch = model.predict(np.repeat(batch, n_items), np.tile(np.arange(n_items), len(batch)), item_features=features, num_threads=NUM_THREADS).reshape(len(batch), n_items)

        for i, u in enumerate(batch):
            scores = scores_batch[i].copy()
            scores[train[u].indices] = -np.inf
            top_k = np.argpartition(-scores, k-1)[:k]

            p_tfidf_idx = [rec_to_tfidf[idx_to_rec[idx]] for idx in top_k if idx_to_rec[idx] in rec_to_tfidf]
            t_tfidf_idx = [rec_to_tfidf[idx_to_rec[idx]] for idx in test[u].indices if idx_to_rec[idx] in rec_to_tfidf]

            if len(p_tfidf_idx) == 0 or len(t_tfidf_idx) == 0: continue

            p_vec, t_vec = tfidf_mat[p_tfidf_idx], tfidf_mat[t_tfidf_idx]
            d_p2t, d_t2p = np.min(euclidean_distances(p_vec, t_vec), axis=1), np.min(euclidean_distances(t_vec, p_vec), axis=1)
            hd_scores.append(max(np.percentile(d_p2t, 95), np.percentile(d_t2p, 95)))

    return np.mean(hd_scores)

hd95 = evaluate_hd95(model, interactions_matrix, test_interactions_csr, tfidf_matrix, item_features_matrix, recipe_id_to_tfidf_idx, idx_to_recipe_id)

# ==============================================================================
# 8. SONUÇLAR VE KAYIT
# ==============================================================================
print("\n" + "="*50)
print("🏆 NİHAİ MODEL PERFORMANS RAPORU (10-CORE & NDCG)")
print("="*50)
print(f"Precision@10: {precision:.4f} (Nokta atışı isabet)")
print(f"Recall@10:    {recall:.4f} (Kullanıcının sevdiklerini yakalama)")
print(f"NDCG@10:      {ndcg:.4f} (Sıralama kalitesi)")
print(f"F1-Score@10:  {f1:.4f} (Dengeli doğruluk skoru)")
print(f"AUC:          {auc:.4f} (Doğru sıralama gücü)")
print(f"Diversity:    {diversity:.4f} (Önerilenlerin kendi içindeki çeşitliliği)")
print(f"Coverage:     {coverage:.4f} (Kataloğun yüzde kaçını keşfetti)")
print(f"HD95:         {hd95:.4f} (TF-IDF anlamsal mesafe)")
print("="*50)


🧪 METRİKLER HESAPLANIYOR (Bu işlem biraz zaman alabilir)...



HD95: 100%|██████████| 36/36 [02:58<00:00,  4.95s/it]


🏆 NİHAİ MODEL PERFORMANS RAPORU (10-CORE & NDCG)
Precision@10: 0.0202 (Nokta atışı isabet)
Recall@10:    0.0301 (Kullanıcının sevdiklerini yakalama)
NDCG@10:      0.0303 (Sıralama kalitesi)
F1-Score@10:  0.0242 (Dengeli doğruluk skoru)
AUC:          0.7853 (Doğru sıralama gücü)
Diversity:    0.8338 (Önerilenlerin kendi içindeki çeşitliliği)
Coverage:     0.3306 (Kataloğun yüzde kaçını keşfetti)
HD95:         1.3538 (TF-IDF anlamsal mesafe)


In [ ]:
# ==================================================
# 🏆 NİHAİ MODEL PERFORMANS RAPORU
# ==================================================
# Precision@10: 0.0202 (Nokta atışı isabet)
# Recall@10:    0.0301 (Kullanıcının sevdiklerini yakalama)
# NDCG@10:      0.0303 (Sıralama kalitesi)
# F1-Score@10:  0.0242 (Dengeli doğruluk skoru)
# AUC:          0.7853 (Doğru sıralama gücü)
# Diversity:    0.8338 (Önerilenlerin kendi içindeki çeşitliliği)
# Coverage:     0.3306 (Kataloğun yüzde kaçını keşfetti)
# HD95:         1.3538 (TF-IDF anlamsal mesafe)
# ==================================================